# Qwen3.5-4B Answer-First Evaluation on Colab

由于本地 ModelScope 模型架构与 unsloth LoRA 不匹配（linear_attn vs self_attn），需要在 Colab 上评估。

**架构差异**:
- `unsloth/Qwen3.5-4B`: 标准 self_attn (q_proj, k_proj, v_proj, o_proj)
- `Qwen/Qwen3.5-4B` (ModelScope): linear_attn (in_proj_qkv, out_proj)

| 配置 | |
|------|-----|
| 测试数据 | 897 条 |
| GPU | T4 (16GB) |
| 预计时间 | ~10分钟 |

In [ ]:
#@title 1. 安装依赖
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy; _numpy = f'numpy=={numpy.__version__}'
    except: _numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}

!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

print("✓ 依赖安装完成")

In [ ]:
#@title 2. 上传测试数据和 LoRA
from google.colab import files
import json
import shutil
from pathlib import Path

print("请上传以下文件:
print("1. test_answer_first.json (测试数据)")
print("2. qwen35-4b-answer-first.zip (LoRA 模型，或单独上传 adapter_model.safetensors)")
uploaded = files.upload()

# 处理上传文件
for fn in uploaded.keys():
    print(f'上传文件: {fn} ({len(uploaded[fn])} bytes)')

# 如果上传了 zip，解压
if 'qwen35-4b-answer-first.zip' in uploaded:
    shutil.unpack_archive('qwen35-4b-answer-first.zip', 'qwen35-4b-answer-first')
    print("✓ LoRA 模型解压完成")

# 验证测试数据
test_file = 'test_answer_first.json'
if test_file in uploaded or Path(test_file).exists():
    with open(test_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"测试数据: {len(data)} 条")
    print(f"格式检查: {data[0]['conversations'][2]['content'][:50]}...")

print("\n✓ 文件验证完成")

In [ ]:
#@title 3. 加载模型 + LoRA
from unsloth import FastLanguageModel
import torch

# 加载基础模型
print("加载模型: unsloth/Qwen3.5-4B")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-4B",
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.9,
)

# 加载 LoRA adapter
lora_path = "qwen35-4b-answer-first"
if Path(lora_path).exists():
    print(f"加载 LoRA: {lora_path}")
    model.load_adapter(lora_path, adapter_name="answer_first")
    print("✓ LoRA 加载成功")
else:
    print("警告: LoRA 路径不存在，使用原始模型")

# 设置推理模式
FastLanguageModel.for_inference(model)

# GPU 信息
gpu_stats = torch.cuda.get_device_properties(0)
print(f"\nGPU = {gpu_stats.name}. Total memory = {gpu_stats.total_memory / 1024**3:.1f} GB")
print("\n✓ 模型加载完成")

In [ ]:
#@title 4. 运行评估
import json
import re
import time
import gc
from tqdm import tqdm
from collections import Counter

# 解析答案优先格式输出
def extract_sentiment(text: str) -> int:
    text = text.replace('<|channel>thought', '').replace('<channel|>', '')
    match = re.search(r'"sentiment":\s*([0-2])', text)
    if match:
        return int(match.group(1))
    match = re.search(r'^([0-2])\s*', text.strip())
    if match:
        return int(match.group(1))
    return -1

# 系统提示
system_prompt = """You are a professional e-commerce review sentiment analysis expert.

Analyze the review and classify the sentiment as:
- 0: Negative
- 1: Neutral
- 2: Positive

Output your answer as JSON: {"sentiment": 0/1/2}"""

# 加载测试数据
with open('test_answer_first.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 限制样本数量 (可选)
SAMPLES = len(data)  #@param {type:"integer"}
if SAMPLES < len(data):
    data = data[:SAMPLES]

print(f"测试数据: {len(data)} 条")
print(f"\n开始推理 (答案优先格式，max_tokens=20)...")

# 逐条推理
results = []
start_time = time.time()

for i, item in enumerate(tqdm(data, desc="推理")): 
    convs = item["conversations"]
    user_content = convs[1]["content"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    true_label = item.get("label", -1)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs['input_ids'].shape[1]
    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    pred_label = extract_sentiment(text)

    results.append({
        "true": true_label,
        "pred": pred_label,
        "correct": pred_label == true_label,
        "raw": text[:100],
    })

    # 定期清理内存
    if (i + 1) % 50 == 0:
        gc.collect()
        torch.cuda.empty_cache()

infer_time = time.time() - start_time
print(f"\n推理完成，耗时: {infer_time:.1f}秒")

In [ ]:
#@title 5. 计算结果
# 计算准确率
valid = len([r for r in results if r["pred"] != -1])
correct = len([r for r in results if r["correct"]])
parse_errors = len([r for r in results if r["pred"] == -1])
accuracy = correct / valid * 100 if valid > 0 else 0

# 混淆矩阵
cm = Counter()
for r in results:
    if r['pred'] != -1:
        cm[(r['true'], r['pred'])] += 1

print("="*60)
print("评估结果")
print("="*60)
print(f"准确率: {accuracy:.2f}%")
print(f"总样本: {len(results)}")
print(f"正确: {correct}")
print(f"解析错误: {parse_errors}")
print(f"推理速度: {len(results)/infer_time:.2f} 条/秒")

print(f"\n各类召回率:")
for label, name in enumerate(['负面', '中性', '正面']):
    total = sum(cm[(label, p)] for p in [0, 1, 2])
    correct_l = cm[(label, label)]
    recall = correct_l / total * 100 if total > 0 else 0
    print(f"  {name}: {recall:.1f}% ({correct_l}/{total})")

print(f"\n对比 Qwen3-4B:")
print(f"  Qwen3-4B 固定温度: 80.38%")
print(f"  Qwen3-4B 动态温度: 80.71%")
print(f"  Qwen3.5-4B: {accuracy:.2f}%")

if accuracy > 80.71:
    print(f"\n  ✓ Qwen3.5 超越 Qwen3 (+{accuracy - 80.71:.2f}%)")
else:
    print(f"\n  ! Qwen3.5 未超越 Qwen3 ({accuracy - 80.71:.2f}%)")

In [ ]:
#@title 6. 保存并下载结果
result_file = "qwen35_answer_first_eval.json"

with open(result_file, "w", encoding="utf-8") as f:
    json.dump({
        "model": "Qwen3.5-4B",
        "format": "answer_first",
        "accuracy": accuracy,
        "total": len(results),
        "correct": correct,
        "parse_errors": parse_errors,
        "speed": len(results)/infer_time,
        "recall": {
            "negative": cm[(0, 0)] / sum(cm[(0, p)] for p in [0, 1, 2]) * 100 if sum(cm[(0, p)] for p in [0, 1, 2]) > 0 else 0,
            "neutral": cm[(1, 1)] / sum(cm[(1, p)] for p in [0, 1, 2]) * 100 if sum(cm[(1, p)] for p in [0, 1, 2]) > 0 else 0,
            "positive": cm[(2, 2)] / sum(cm[(2, p)] for p in [0, 1, 2]) * 100 if sum(cm[(2, p)] for p in [0, 1, 2]) > 0 else 0,
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"结果已保存: {result_file}")

# 下载结果文件
from google.colab import files
files.download(result_file)

print("\n✓ 评估完成")

## 评估完成

结果已下载到本地，后续步骤：

1. 将结果文件放入 `6_experiments_results/` 目录
2. 更新实验报告
3. 对比不同模型的准确率